In [0]:
import time
from pyspark.sql.functions import *
from pyspark.sql.types import *

kafka_server = "6.tcp.eu.ngrok.io:14538"

output_path = "/Volumes/workspace/default/my_volume/weather_final_output"
checkpoint_path = "/Volumes/workspace/default/my_volume/checkpoints/weather_final"

# Nested JSON schema
raw_schema = StructType([
    StructField("location", StructType([
        StructField("name", StringType()),
        StructField("localtime", StringType())
    ])),
    StructField("current", StructType([
        StructField("temp_c", DoubleType()),
        StructField("humidity", IntegerType()),
        StructField("wind_kph", DoubleType()),
        StructField("last_updated", StringType()),
    ]))
])

batch_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_server) \
    .option("subscribe", "weather_reports") \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), raw_schema).alias("data")) \
    .select(
        col("data.location.name").alias("City"),
        col("data.current.temp_c").alias("Temperature"),
        col("data.current.humidity").alias("Humidity"),
        col("data.current.wind_kph").alias("Wind_speed"),
        col("data.location.localtime").alias("Local_time"),
        col("data.current.last_updated").alias("Last_updated")
    ) \
    .withColumn("timestamp", current_timestamp()) \
    .filter(col("City").isNotNull()) \
    .filter(col("Temperature").isNotNull())

avg_temp_df = batch_df \
    .groupBy(
        window(col("timestamp"), "5 minutes", "1 minute"),
        col("City")
    ).agg(avg("Temperature").alias("avg_temperature"))

# Eski output temizle
dbutils.fs.rm(output_path, recurse=True)

avg_temp_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

spark.sql(f"SELECT * FROM delta.`{output_path}`").show()

+--------------------+------+------------------+
|              window|  City|   avg_temperature|
+--------------------+------+------------------+
|{2026-05-15 17:57...|Ankara|11.138412698412605|
|{2026-05-15 17:58...|Ankara|11.138412698412605|
|{2026-05-15 17:59...|Ankara|11.138412698412605|
|{2026-05-15 17:56...|Ankara|11.138412698412605|
|{2026-05-15 17:55...|Ankara|11.138412698412605|
+--------------------+------+------------------+



In [0]:
# Eski output'u temizle
dbutils.fs.rm("/Volumes/workspace/default/my_volume/weather_final_output", recurse=True)
dbutils.fs.rm("/Volumes/workspace/default/my_volume/checkpoints/weather_final", recurse=True)

output_path = "/Volumes/workspace/default/my_volume/weather_final_output"
checkpoint_path = "/Volumes/workspace/default/my_volume/checkpoints/weather_final"

# parsed_df'i batch olarak oku (streaming değil)
from pyspark.sql.functions import window, avg, col, current_timestamp, from_json
from pyspark.sql.types import *

schema = StructType([
    StructField("City", StringType()),
    StructField("Temperature", DoubleType()),
    StructField("Humidity", IntegerType()),
    StructField("Wind_speed", DoubleType()),
    StructField("Local_time", StringType()),
    StructField("Last_updated", StringType())
])

kafka_server = "6.tcp.eu.ngrok.io:14538"

batch_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_server) \
    .option("subscribe", "weather_reports") \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("timestamp", current_timestamp())

avg_temp_df = batch_df \
    .groupBy(
        window(col("timestamp"), "5 minutes", "1 minute"),
        col("City")
    ).agg(avg("Temperature").alias("avg_temperature"))

avg_temp_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

spark.sql(f"SELECT * FROM delta.`{output_path}`").show()

+--------------------+---------+------------------+
|              window|     City|   avg_temperature|
+--------------------+---------+------------------+
|{2026-05-15 17:59...|Amsterdam|10.152830188679237|
|{2026-05-15 17:55...|Amsterdam|10.152830188679237|
|{2026-05-15 17:58...|     NULL|              NULL|
|{2026-05-15 17:56...|     NULL|              NULL|
|{2026-05-15 17:56...|Amsterdam|10.152830188679237|
|{2026-05-15 17:55...|     NULL|              NULL|
|{2026-05-15 17:57...|Amsterdam|10.152830188679237|
|{2026-05-15 17:59...|     NULL|              NULL|
|{2026-05-15 17:57...|     NULL|              NULL|
|{2026-05-15 17:58...|Amsterdam|10.152830188679237|
+--------------------+---------+------------------+



In [0]:
avg_temp_df = batch_df \
    .filter(col("City").isNotNull()) \
    .filter(col("Temperature").isNotNull()) \
    .groupBy(
        window(col("timestamp"), "5 minutes", "1 minute"),
        col("City")
    ).agg(avg("Temperature").alias("avg_temperature"))

avg_temp_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

spark.sql(f"SELECT * FROM delta.`{output_path}`").show()

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-5129100530751367>, line 14
      1 avg_temp_df = batch_df \
      2     .filter(col("City").isNotNull()) \
      3     .filter(col("Temperature").isNotNull()) \
   (...)
      6         col("City")
      7     ).agg(avg("Temperature").alias("avg_temperature"))
      9 avg_temp_df.write \
     10     .format("delta") \
     11     .mode("overwrite") \
     12     .save(output_path)
---> 14 spark.sql(f"SELECT * FROM delta.`{output_path}`").show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in prope